In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# controllo variabili Dummies 

In [2]:
dataset = client.download_file('cleaned/merged/combination/subMERGE_4-3_elecsys_FBP.csv')
df_real = dataset.copy(deep=True)

In [3]:
df_real.columns

Index(['RID', 'COHORT', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'FLDSTRENG',
       'FSVERSION', 'IMAGEUID', 'update_stamp', 'Ventricles%ICV',
       'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV',
       'STATUS', 'ADAS11', 'ADAS13', 'CDRGLOB', 'CDRSB', 'FAQ', 'MMSE', 'MOCA',
       'RAVLT_immediate', 'AB40_CSF', 'AB4240_CSF', 'AB42_CSF', 'METHOD_CSF',
       'PT181_AB42_CSF', 'PT181_CSF', 'TTAU_AB42_CSF', 'TTAU_CSF',
       'ENTORHINAL_SUVR', 'INFERIOR_TEMPORAL_SUVR', 'METHOD_PET',
       'SUMMARY_SUVR', 'TAU_METAROI', 'TRACER', 'AGE', 'APOE', 'APOE_4',
       'DX/CN', 'DX/Dementia', 'DX/MCI', 'EDUCAT', 'ETHNICITY/latino',
       'ETHNICITY/not_latino', 'GENDER/female', 'GENDER/male',
       'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed',
       'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american',
       'RACE/White'],
      dtype='object')

In [ ]:
[x for x in df_real.columns if '/' in x]

In [ ]:
list_dummies = [x for x in df_real.columns if '/' in x]
list_cat_dummies = list(set([x.split('/')[0] for x in list_dummies]))
# Funzione per trovare soggetti con valori contrastanti per una categoria
def find_conflicting_subjects(df, dummy_cols):
    """
    Trova RID con valori contrastanti:
    - Tutte le colonne dummy = 0 (nessuna categoria selezionata)
    - Più di una colonna dummy = 1 (categorie multiple selezionate)
    Esclude righe dove tutte le colonne dummy sono NaN.
    """
    
    # Converti a numerico (gestisce stringhe 'True', 'False', '0', '1')
    df_dummies = df[['RID', 'EXAMDATE'] + dummy_cols].copy()
    print(dummy_cols)
    for col in dummy_cols:
        df_dummies[col] = pd.to_numeric(
            df_dummies[col].replace({'True': 1, 'False': 0, 'true': 1, 'false': 0}), 
            errors='coerce'
        )
    
    # Escludi righe dove TUTTE le colonne dummy sono NaN
    all_nan_mask = df_dummies[dummy_cols].isna().all(axis=1)
    df_dummies = df_dummies[~all_nan_mask]
    
    # Calcola la somma delle dummy per ogni riga (skipna=True è default)
    df_dummies['sum_dummies'] = df_dummies[dummy_cols].sum(axis=1)
    
    # Casi contrastanti: somma = 0 (tutti zero) o somma > 1 (più di un 1)
    all_zeros = df_dummies[df_dummies['sum_dummies'] == 0]
    multiple_ones = df_dummies[df_dummies['sum_dummies'] > 1]
    return {
        'all_zeros': all_zeros[['RID', 'EXAMDATE']],
        'multiple_ones': multiple_ones['RID'].unique().tolist(),
        'all_zeros_count': len(all_zeros['RID'].unique()),
        'multiple_ones_count': len(multiple_ones['RID'].unique())
    }
    
# Analisi per ogni categoria
results = {}
print("=" * 80)
print("ANALISI VALORI CONTRASTANTI NELLE DUMMIES")
print("=" * 80)

for cat_name in list_cat_dummies:
    # Verifica che le colonne esistano nel dataframe
    existing_cols = [col for col in list_dummies if cat_name in col and col in df_real.columns]
    if not existing_cols:
        print(f"\n{cat_name}: Colonne non trovate nel dataframe")
        continue
    
    result = find_conflicting_subjects(df_real, existing_cols)
    results[cat_name] = result
    
    print(f"\n{'─' * 60}")
    print(f"CATEGORIA: {cat_name}")
    print(f"Colonne: {existing_cols}")
    print(f"{'─' * 60}")
    
    print(f"\n  ❌ RID con TUTTI ZERI (nessuna categoria): {result['all_zeros_count']}")
    if result['all_zeros_count'] > 0:
        if result['all_zeros_count'] <= 20:
            print(f"     RID: {result['all_zeros']}")
        else:
            print(f"     Primi 20 RID: {result['all_zeros'][:20]}")
            print(f"     ... e altri {result['all_zeros_count'] - 20}")
    
    print(f"\n  ⚠️  RID con PIÙ DI UN 1 (categorie multiple): {result['multiple_ones_count']}")
    if result['multiple_ones_count'] > 0:
        if result['multiple_ones_count'] <= 20:
            print(f"     RID: {result['multiple_ones']}")
        else:
            print(f"     Primi 20 RID: {result['multiple_ones'][:20]}")
            print(f"     ... e altri {result['multiple_ones_count'] - 20}")

# Riepilogo finale
print(f"\n{'=' * 80}")
print("RIEPILOGO")
print(f"{'=' * 80}")

summary_data = []
for cat_name, result in results.items():
    summary_data.append({
        'Categoria': cat_name,
        'RID tutti zeri': result['all_zeros_count'],
        'RID più di un 1': result['multiple_ones_count'],
        'Totale contrastanti': result['all_zeros_count'] + result['multiple_ones_count']
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

In [ ]:
search = client.query_files(#query={'custom.level' : 'merged', 'custom.file_code' :'COFMERGE'})
    query={'custom.level' : 'cleaned_02', 'custom.file_code' :'PTDEMOG'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

df_demog = zip_files[list(zip_files.keys())[0]].copy(deep=True)

In [ ]:
sub = results['MARRY']['all_zeros']['RID'].unique()
print('soggetti: ', len(sub))
print('righe: ', len(results['MARRY']['all_zeros']))

In [ ]:
df_real.columns

In [ ]:
df_real[df_real['RID'].isin(sub)][['RID', 'EXAMDATE', 'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed']]

In [ ]:
df_demog.columns

In [ ]:
df_demog[df_demog['RID'].isin(sub)][['RID', 'EXAMDATE', 'MARRY/divorced', 'MARRY/married', 'MARRY/single',
       'MARRY/widowed']]

In [ ]:
search = client.query_files(#query={'custom.level' : 'merged', 'custom.file_code' :'COFMERGE'})
    query={'custom.level' : 'raw', 'custom.file_code' :'PTDEMOG'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

df_raw = zip_files[list(zip_files.keys())[0]].copy(deep=True)

In [ ]:
df_raw['PTRACCAT'].value_counts()

In [ ]:
lsub = [2022,2072,6203,6241,6254,6517,6785,6952,7105,7105]

In [ ]:
df_raw[df_raw['RID'].isin(lsub)][['RID', 'VISDATE', 'PTRACCAT']].sort_values(by='RID')

In [ ]:
df_demog[df_demog['RID'].isin(null_demog)][['RID', 'VISDATE', 'PTETHCAT']]

In [ ]:
check_sub = list(set(null_demog) - set(sub))

df_real[df_real['RID'].isin(check_sub)][['RID', 'EXAMDATE', 'ETHNICITY/latino', 'ETHNICITY/not_latino']]

In [ ]:
df_real[df_real['RID']==10863][['RID', 'EXAMDATE', 'ETHNICITY/latino', 'ETHNICITY/not_latino']]

In [ ]:
df_demog[df_demog['RID'].isin(sub)][['RID', 'VISDATE', 'PTETHCAT']]

In [ ]:
for cat in list_cat_dummies:
    sub_cat = results[cat]['all_zeros']
        # Conta quanti soggetti di sub_cat sono presenti in df_demog
    presenti = df_demog[df_demog['RID'].isin(sub_cat)]['RID'].nunique()
    totale = len(sub_cat)

    if presenti > 0:
        print(f"CATEGORIA: {cat}\nSoggetti presenti in df_demog: {presenti} su {totale} ({presenti/totale*100:.1f}%)\n--------------------------------------------------------------------------------------------------------------------------------")
    else:
        print(f"CATEGORIA: {cat}\nNessun soggetto presente in df_demog\n--------------------------------------------------------------------------------------------------------------------------------")


In [ ]:
sub_dx = results['DX']['all_zeros']
list_dx = list(df_dx[df_dx['RID'].isin(sub_dx)]['RID'].unique())
list_adni = list(df_adni[df_adni['RID'].isin(sub_dx)]['RID'].unique())

list_sum = list(set(list_dx) | set(list_adni))


print('dx: ',len(list_dx))
print('adni: ',len(list_adni))
print('sum: ',len(list_sum))
print('sub_tot: ',len(sub_dx))


In [ ]:
df_dx[df_dx['RID'].isin(sub_dx)][['RID', 'EXAMDATE', 'DX/CN', 'DX/Dementia', 'DX/MCI']]

In [ ]:
sub_eth = results['ETHNICITY']['all_zeros']

In [ ]:
df_real[df_real['RID']==sub_eth['RID'][0]][['RID', 'EXAMDATE', 'DX/CN', 'DX/Dementia', 'DX/MCI']]

In [ ]:
df_real['DX/Dementia'].value_counts()

In [ ]:
df_dx[df_dx['RID'].isin(sub_dx)]['DX'].unique()

In [ ]:
df_adni[df_adni['RID'].isin(sub_dx)]['DX'].unique()

In [ ]:
df_demog.columns

In [ ]:
df_demog['DIAGNOSIS'].unique()

In [ ]:
cat = 'DX'
sub = results[cat]['all_zeros']
spec = df_demog[df_demog['RID'].isin(sub)][['RID', 'DIAGNOSIS']]

In [ ]:
len(spec['RID'].unique())


In [ ]:
df_real.columns

In [ ]:
df_real[df_real['RID'].isin(sub)][['RID', 'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american',
       'RACE/White']]

# Verifica dataset generato e dataset originale

In [ ]:
df_gen = client.download_file('synthetic/synthetic_data_generation_1.csv')
df_gen.columns

In [ ]:
df_gen.describe()

In [ ]:
m = client.get_metadata('synthetic/synthetic_data_generation_1.csv')
metadata = m['metadata']['custom']
metadata

In [ ]:
df_real = client.download_file('cleaned/merged/combination/subMERGE_4-3_elecsys_FBP.csv')
df_real[['Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV',
       'Fusiform%ICV', 'MidTemp%ICV', 'MMSE', 'RAVLT_immediate', 'FAQ', 'MOCA',
       'CDRSB', 'CDRGLOB', 'ADAS11', 'ADAS13', 'AB40_CSF', 'AB42_CSF',
       'AB4240_CSF', 'PT181_CSF', 'TTAU_CSF', 'PT181_AB42_CSF',
       'TTAU_AB42_CSF', 'SUMMARY_SUVR', 'TAU_METAROI',
       'INFERIOR_TEMPORAL_SUVR', 'ENTORHINAL_SUVR', 'DX/CN',
       'DX/Dementia', 'DX/MCI']].describe()

# basic test precedenti

In [ ]:
search = client.query_files(
    query={'custom.level' : 'merged_comb', 'custom.file_code' : 'subMERGE_4-3_elecsys_FBP_synthetic_reversed_missing_injected'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

In [ ]:
search['included_files'][0]['object_name']

In [ ]:
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df = dataset.copy(deep=True)

In [ ]:
df.columns

In [ ]:
df['DX/CN'].unique()

In [ ]:
display(df.head())

In [ ]:
nan_df = df.isna().sum().to_frame('NaN Count')
nan_df['% NaN'] = (df.isna().sum() / len(df) * 100).round(2)

pd.reset_option('display.max_rows')
display(nan_df)

In [ ]:
df[['AB40_CSF', 'AB4240_CSF', 'AB42_CSF', 'PT181_AB42_CSF', 'PT181_CSF', 'TTAU_AB42_CSF',
       'TTAU_CSF']].isna().sum()

In [ ]:
df.columns

In [ ]:
col_del = ['ICV%ICV', 'AMY_CENTILOIDS', 'AGE_AD_BEG', 'AGE_AD_DX', 'AGE_COG_BEG', 'AGE_bl', 'Tprofile', 'Aprofile', 'DIAN_MUTATION']

In [ ]:
df.drop(columns=col_del, inplace=True)
df.columns


In [ ]:
volumi = ['Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV']
csf = ['AB40_CSF', 'AB4240_CSF', 'AB42_CSF', 'PT181_AB42_CSF', 'PT181_CSF', 'TTAU_AB42_CSF', 'TTAU_CSF']
pet = ['CTX_ENTORHINAL_SUVR','CTX_FUSIFORM_SUVR', 'CTX_INFERIORPARIETAL_SUVR', 'CTX_LATERALOCCIPITAL_SUVR', 'CTX_MIDDLETEMPORAL_SUVR',
       'CTX_PARAHIPPOCAMPAL_SUVR', 'ENTORHINAL_SUVR', 'INFERIOR_TEMPORAL_SUVR', 'SUMMARY_SUVR', 'TAU_METAROI']

In [ ]:
# Verifica se una riga ha almeno un valore valido per categoria
has_volumi = df[volumi].notna().any(axis=1)
has_csf = df[csf].notna().any(axis=1)
has_pet = df[pet].notna().any(axis=1)

# Cancella righe dove TUTTE e tre le categorie sono completamente NaN
mask_valid = has_volumi | has_csf | has_pet
rows_before = len(df)
df = df[mask_valid]
rows_after = len(df)

print(f"Righe eliminate: {rows_before - rows_after}")
print(f"Righe rimanenti: {rows_after}")


In [ ]:
# Statistiche (non inclusive)
# Ricalcola dopo la pulizia
has_volumi = df[volumi].notna().any(axis=1)
has_csf = df[csf].notna().any(axis=1)
has_pet = df[pet].notna().any(axis=1)

total = len(df)

# Solo due categorie (escludendo la terza)
vol_csf_only = (has_volumi & has_csf & ~has_pet).sum()
vol_pet_only = (has_volumi & has_pet & ~has_csf).sum()
csf_pet_only = (has_csf & has_pet & ~has_volumi).sum()

# Tutte e tre
all_three = (has_volumi & has_csf & has_pet).sum()

# Solo una categoria
vol_only = (has_volumi & ~has_csf & ~has_pet).sum()
csf_only = (has_csf & ~has_volumi & ~has_pet).sum()
pet_only = (has_pet & ~has_volumi & ~has_csf).sum()

print(f"\n--- Statistiche (su {total} righe) ---")
print(f"Solo Volumi:        {vol_only:4d} ({vol_only/total*100:.1f}%)")
print(f"Solo CSF:           {csf_only:4d} ({csf_only/total*100:.1f}%)")
print(f"Solo PET:           {pet_only:4d} ({pet_only/total*100:.1f}%)")
print(f"Volumi + CSF:       {vol_csf_only:4d} ({vol_csf_only/total*100:.1f}%)")
print(f"Volumi + PET:       {vol_pet_only:4d} ({vol_pet_only/total*100:.1f}%)")
print(f"CSF + PET:          {csf_pet_only:4d} ({csf_pet_only/total*100:.1f}%)")
print(f"Tutti e tre:        {all_three:4d} ({all_three/total*100:.1f}%)")


In [ ]:
# Statistiche (non inclusive)
# Ricalcola dopo la pulizia
has_volumi = df[volumi].notna().any(axis=1)
has_csf = df[csf].notna().any(axis=1)
has_pet = df[pet].notna().any(axis=1)

total = len(df)

# Solo una categoria
vol_only = (has_volumi & ~has_csf & ~has_pet).sum()
csf_only = (has_csf & ~has_volumi & ~has_pet).sum()
pet_only = (has_pet & ~has_volumi & ~has_csf).sum()

# Solo due categorie (escludendo la terza)
vol_csf_only = (has_volumi & has_csf & ~has_pet).sum()
vol_pet_only = (has_volumi & has_pet & ~has_csf).sum()
csf_pet_only = (has_csf & has_pet & ~has_volumi).sum()

# Tutte e tre
all_three = (has_volumi & has_csf & has_pet).sum()

print(f"\n--- Statistiche (su {total} righe) ---")
print(f"Solo Volumi:        {vol_only:4d} ({vol_only/total*100:.1f}%)")
print(f"Solo CSF:           {csf_only:4d} ({csf_only/total*100:.1f}%)")
print(f"Solo PET:           {pet_only:4d} ({pet_only/total*100:.1f}%)")
print(f"Volumi + CSF:       {vol_csf_only:4d} ({vol_csf_only/total*100:.1f}%)")
print(f"Volumi + PET:       {vol_pet_only:4d} ({vol_pet_only/total*100:.1f}%)")
print(f"CSF + PET:          {csf_pet_only:4d} ({csf_pet_only/total*100:.1f}%)")
print(f"Tutti e tre:        {all_three:4d} ({all_three/total*100:.1f}%)")

# Grafico a torta
labels = ['Solo Volumi', 'Solo CSF', 'Solo PET', 
          'Volumi + CSF', 'Volumi + PET', 'CSF + PET', 
          'Tutti e tre']
values = [vol_only, csf_only, pet_only, 
          vol_csf_only, vol_pet_only, csf_pet_only, 
          all_three]
colors = ['#ff9999', '#66b3ff', '#99ff99', 
          '#ffcc99', '#ff99cc', '#99ccff', 
          '#c2c2f0']

# Rimuovi categorie con 0 valori per grafico più pulito
filtered = [(l, v, c) for l, v, c in zip(labels, values, colors) if v > 0]
if filtered:
    labels_f, values_f, colors_f = zip(*filtered)
else:
    labels_f, values_f, colors_f = labels, values, colors

fig, ax = plt.subplots(figsize=(10, 7))
wedges, texts, autotexts = ax.pie(values_f, labels=labels_f, colors=colors_f,
                                   autopct=lambda p: f'{p:.1f}%\n({int(p*total/100)})',
                                   startangle=90, pctdistance=0.75)
ax.set_title(f'Distribuzione dati disponibili per categoria\n(Totale: {total} righe)')
plt.tight_layout()
plt.show()


In [ ]:
len(df)

In [ ]:
# Converte in DataFrame per visualizzazione migliore
nan_df = df.isna().sum().to_frame('NaN Count')
nan_df['% NaN'] = (df.isna().sum() / len(df) * 100).round(2)

pd.set_option('display.max_rows', None)
display(nan_df)


In [ ]:
df.columns

In [ ]:
col_del = ['CTX_ENTORHINAL_SUVR', 'CTX_FUSIFORM_SUVR', 'CTX_INFERIORPARIETAL_SUVR',
       'CTX_LATERALOCCIPITAL_SUVR', 'CTX_MIDDLETEMPORAL_SUVR',
       'CTX_PARAHIPPOCAMPAL_SUVR']
df.drop(columns=col_del, inplace=True)

In [ ]:
new_file_name = 'subMERGE_4-3_elecsys_FBP.csv'
print('New merge file name: ', new_file_name)
file_code = 'subMERGE_4-3_elecsys_FBP'

metadata = client.get_metadata(object_name=search['included_files'][0]['object_name'])
metadata_updated = metadata['metadata']['custom']
metadata_updated.keys()

In [ ]:
# Lista delle colonne attuali di df
current_cols = set(df.columns)

# Keys con liste
list_keys = ['cofattori', 'cofattori_metadata', 'norm_scala', 'norm_volume', 'predittori']

# Keys con dizionari
dict_keys = ['norm_scale_value', 'volume_norm_values']

# Aggiorna le liste: mantieni solo elementi presenti in df.columns
for key in list_keys:
    if key in metadata_updated:
        metadata_updated[key] = [col for col in metadata_updated[key] if col in current_cols]

# Aggiorna i dizionari: mantieni solo chiavi presenti in df.columns
for key in dict_keys:
    if key in metadata_updated:
        metadata_updated[key] = {k: v for k, v in metadata_updated[key].items() if k in current_cols}

# Verifica risultato
print("--- Elementi rimasti per ogni key ---")
for key in list_keys + dict_keys:
    if key in metadata_updated:
        n = len(metadata_updated[key])
        print(f"{key}: {n} elementi")



In [ ]:
metadata_updated['file_code'] = file_code



In [ ]:

# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=df,
    object_name=new_file_name,
    prefix='cleaned/merged/combination',
    metadata=metadata_updated
)